## Gendered Word Count Analysis Notebook

This notebook uses the data from the count_chat_words.py script, and cannot be run without it. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, mannwhitneyu, levene
from statsmodels.stats.proportion import proportions_ztest

# --- Load the enriched gender word count dataset ---
df = pd.read_csv("df_cgw_genderword_counts_with_groups.csv")

# --- Filter groups for comparison ---
group_2022 = df[df["group"] == "2022"]
group_recent = df[df["group"] == "2024/2025"]

# --- Comparison helper for continuous variables ---
def compare_metric(var):
    x = group_2022[var].dropna()
    y = group_recent[var].dropna()

    # Levene's test (variance equality)
    lev_stat, lev_p = levene(x, y)

    # Welch’s t-test
    t_stat, t_p = ttest_ind(x, y, equal_var=False)

    # Mann-Whitney U test
    u_stat, u_p = mannwhitneyu(x, y, alternative="two-sided")

    # Cohen's d
    def cohens_d(a, b):
        pooled_sd = np.sqrt(((np.std(a, ddof=1) ** 2 + np.std(b, ddof=1) ** 2) / 2))
        return (np.mean(a) - np.mean(b)) / pooled_sd

    d = cohens_d(x, y)

    print(f"\n📊 Comparing `{var}` (2022 vs. 2024/2025)")
    print(f"  Levene’s test:     W = {lev_stat:.3f}, p = {lev_p:.4f}")
    print(f"  Welch’s t-test:    t = {t_stat:.3f}, p = {t_p:.4f}")
    print(f"  Mann-Whitney U:    U = {u_stat:.0f}, p = {u_p:.4f}")
    print(f"  Cohen’s d:         d = {d:.3f}")

    # --- Plot ---
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df[df[var].notna()], x="group", y=var, palette="Set2")
    plt.title(f"{var.replace('_', ' ').title()} by Posting Group")
    plt.xlabel("Period")
    plt.ylabel(var.replace('_', ' ').title())
    plt.grid(True, axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# --- Run comparisons and plots for continuous metrics ---
compare_metric("fem_ratio")
compare_metric("masc_ratio")
compare_metric("gendered_ratio")

# --- Comparison helper for proportions ---
def compare_proportions(var):
    count = [
        group_2022[var].sum(),
        group_recent[var].sum()
    ]
    nobs = [len(group_2022), len(group_recent)]

    z_stat, p_val = proportions_ztest(count, nobs)

    print(f"\n📊 Z-test for proportion of `{var}`:")
    print(f"  Proportion 2022:        {count[0] / nobs[0]:.3f}")
    print(f"  Proportion 2024/2025:   {count[1] / nobs[1]:.3f}")
    print(f"  z = {z_stat:.3f}, p = {p_val:.4f}")

# --- Run proportion tests ---
compare_proportions("has_feminine")
compare_proportions("has_masculine")

# --- Plot bar chart for has_feminine and has_masculine ---
prop_df = df.groupby("group")[["has_feminine", "has_masculine"]].mean().reset_index()
prop_df_melted = prop_df.melt(id_vars="group", var_name="Word Type", value_name="Proportion")

plt.figure(figsize=(8, 5))
sns.barplot(data=prop_df_melted, x="group", y="Proportion", hue="Word Type", palette="Set2")
plt.title("Proportion of Ads Using Feminine vs. Masculine Words")
plt.ylabel("Proportion of Ads")
plt.xlabel("Period")
plt.ylim(0, 1)
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
